In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

RUN_ID = "manual_001"

print("Namespace:", NS)

### Load Bronze sensor data

In [0]:
bronze_sensor = spark.table(
    f"{NS}.bronze_sensor_events"
)

display(bronze_sensor)

print(
    "Bronze sensor count:",
    bronze_sensor.count()
)

In [0]:
sensor_checked = (
    bronze_sensor

    .withColumn(
        "temperature_numeric",
        F.expr(
            "try_cast(temperature_c AS DOUBLE)"
        )
    )

    .withColumn(
        "event_timestamp",
        F.expr(
            "try_cast(event_time AS TIMESTAMP)"
        )
    )
)

In [0]:
display(
    sensor_checked.select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_timestamp",
        "temperature_c",
        "temperature_numeric"
    )
)

### Verify "hot"

In [0]:
display(
    sensor_checked
    .filter(
        F.col("temperature_c") == "hot"
    )
    .select(
        "event_id",
        "temperature_c",
        "temperature_numeric"
    )
)

In [0]:
bronze_shipments = spark.table(
    f"{NS}.bronze_shipments"
)

valid_shipment_ids = (
    bronze_shipments
    .select("shipment_id")
    .distinct()
)

display(valid_shipment_ids)

In [0]:
sensor_checked = (
    sensor_checked.alias("s")
    .join(
        valid_shipment_ids.alias("sh"),
        F.col("s.shipment_id")
        == F.col("sh.shipment_id"),
        "left"
    )
    .select(
        "s.*",
        F.col("sh.shipment_id")
        .alias("matched_shipment_id")
    )
)

In [0]:
display(
    sensor_checked.select(
        "event_id",
        "shipment_id",
        "matched_shipment_id"
    )
)

In [0]:
validated_sensor = (
    sensor_checked
    .withColumn(
        "rejection_reason",

        F.when(
            F.col("event_id").isNull(),
            F.lit("missing_event_id")
        )

        .when(
            F.col("shipment_id").isNull(),
            F.lit("missing_shipment_id")
        )

        .when(
            F.col("event_timestamp").isNull(),
            F.lit("invalid_event_time")
        )

        .when(
            F.col("temperature_numeric").isNull(),
            F.lit("nonnumeric_temperature")
        )

        .when(
            F.col("matched_shipment_id").isNull(),
            F.lit("unknown_shipment")
        )

        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

In [0]:
display(
    validated_sensor.select(
        "event_id",
        "shipment_id",
        "temperature_c",
        "rejection_reason"
    )
)

In [0]:
rejected = (
    validated_sensor
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

display(
    rejected.select(
        "event_id",
        "shipment_id",
        "temperature_c",
        "rejection_reason"
    )
)

### Separate valid from invalid

In [0]:
valid_sensor = (
    validated_sensor
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_sensor = (
    validated_sensor
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

In [0]:
print(
    "Bronze:",
    bronze_sensor.count()
)

print(
    "Valid before dedup:",
    valid_sensor.count()
)

print(
    "Quarantine:",
    quarantine_sensor.count()
)

In [0]:
bronze_count = bronze_sensor.count()
valid_count = valid_sensor.count()
quarantine_count = quarantine_sensor.count()

assert (
    valid_count + quarantine_count
    == bronze_count
), "Silver reconciliation failed"

print("Reconciliation passed ✅")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dedup_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("_processing_time").asc(),
        F.col("_source_file").asc()
    )
)

In [0]:
valid_ranked = (
    valid_sensor
    .withColumn(
        "_duplicate_rank",
        F.row_number().over(dedup_window)
    )
)

In [0]:
display(
    valid_ranked
    .filter(F.col("event_id") == "E015")
    .select(
        "event_id",
        "shipment_id",
        "sensor_id",
        "event_time",
        "temperature_c",
        "_duplicate_rank"
    )
)

In [0]:
silver_sensor = (
    valid_ranked
    .filter(F.col("_duplicate_rank") == 1)
)

In [0]:
print(
    "Valid records before dedup:",
    valid_sensor.count()
)

print(
    "Valid records after dedup:",
    silver_sensor.count()
)

In [0]:
print(
    "E015 surviving records:",
    silver_sensor
    .filter(F.col("event_id") == "E015")
    .count()
)

In [0]:
silver_sensor_final = (
    silver_sensor
    .select(
        F.col("event_id"),
        F.col("shipment_id"),
        F.col("sensor_id"),

        F.col("event_timestamp")
        .alias("event_time"),

        F.col("temperature_numeric")
        .alias("temperature_c"),

        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

In [0]:
display(silver_sensor_final)

In [0]:
silver_sensor_final.printSchema()

In [0]:
(
    silver_sensor_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.silver_sensor_events"
    )
)

print("silver_sensor_events written successfully")

In [0]:
%sql

SHOW TABLES IN training_lab_classic.cold_chain;

In [0]:
%sql

SELECT *
FROM training_lab_classic.cold_chain.silver_sensor_events
ORDER BY event_time;

In [0]:
%sql

SELECT COUNT(*) AS silver_count
FROM training_lab_classic.cold_chain.silver_sensor_events;

In [0]:
quarantine_sensor_final = (
    quarantine_sensor
    .select(
        F.col("event_id"),
        F.col("shipment_id"),
        F.col("sensor_id"),

        # Raw Bronze values deliberately retained
        F.col("event_time"),
        F.col("temperature_c"),

        F.col("rejection_reason"),

        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

In [0]:
display(quarantine_sensor_final)

In [0]:
(
    quarantine_sensor_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.quarantine_sensor_events"
    )
)

print("quarantine_sensor_events written successfully")

In [0]:
%sql

SELECT *
FROM training_lab_classic.cold_chain.quarantine_sensor_events;

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    temperature_c,
    rejection_reason
FROM training_lab_classic.cold_chain.quarantine_sensor_events
WHERE temperature_c = 'hot';

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    temperature_c,
    rejection_reason
FROM training_lab_classic.cold_chain.quarantine_sensor_events
WHERE shipment_id = 'SH999';

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    temperature_c,
    rejection_reason
FROM training_lab_classic.cold_chain.quarantine_sensor_events
WHERE event_id IS NULL;

In [0]:
%sql

SELECT *
FROM training_lab_classic.cold_chain.silver_sensor_events
WHERE event_id = 'E015';

In [0]:
%sql

SELECT *
FROM training_lab_classic.cold_chain.silver_sensor_events
WHERE event_id = 'E015';

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    temperature_c
FROM training_lab_classic.cold_chain.silver_sensor_events
WHERE event_id = 'E004';

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    temperature_c
FROM training_lab_classic.cold_chain.silver_sensor_events
WHERE event_id IN ('E003', 'E007')
ORDER BY event_id;

In [0]:
%sql

SELECT
    rejection_reason,
    COUNT(*) AS record_count
FROM training_lab_classic.cold_chain.quarantine_sensor_events
GROUP BY rejection_reason
ORDER BY rejection_reason;

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    sensor_id,
    event_time,
    temperature_c,
    rejection_reason,
    _source_file,
    _processing_time,
    _run_id
FROM training_lab_classic.cold_chain.quarantine_sensor_events
ORDER BY rejection_reason;

In [0]:
from pyspark.sql import functions as F

NS = "training_lab_classic.cold_chain"

bronze_sensor = spark.table(
    f"{NS}.bronze_sensor_events"
)

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

quarantine_sensor_table = spark.table(
    f"{NS}.quarantine_sensor_events"
)

In [0]:
bronze_count = bronze_sensor.count()

silver_count = silver_sensor_table.count()

quarantine_count = quarantine_sensor_table.count()

print("===== STORED TABLE COUNTS =====")
print("Bronze sensor rows:", bronze_count)
print("Silver sensor rows:", silver_count)
print("Quarantine sensor rows:", quarantine_count)

In [0]:
valid_before_dedup_count = valid_sensor.count()

print(
    "Valid rows before deduplication:",
    valid_before_dedup_count
)


In [0]:
assert (
    valid_before_dedup_count
    + quarantine_count
    == bronze_count
), (
    f"Validation reconciliation failed: "
    f"valid={valid_before_dedup_count}, "
    f"quarantine={quarantine_count}, "
    f"bronze={bronze_count}"
)

print(
    "Validation reconciliation passed:"
)

print(
    f"{valid_before_dedup_count} valid + "
    f"{quarantine_count} quarantine = "
    f"{bronze_count} Bronze"
)

In [0]:
duplicates_removed = (
    valid_before_dedup_count
    - silver_count
)

print("===== DEDUPLICATION CHECK =====")
print(
    "Valid rows before dedup:",
    valid_before_dedup_count
)

print(
    "Silver rows after dedup:",
    silver_count
)

print(
    "Duplicates removed:",
    duplicates_removed
)

In [0]:
reconciled_count = (
    silver_count
    + quarantine_count
    + duplicates_removed
)

print("===== FINAL RECONCILIATION =====")

print(
    "Bronze:",
    bronze_count
)

print(
    "Silver:",
    silver_count
)

print(
    "Quarantine:",
    quarantine_count
)

print(
    "Duplicates removed:",
    duplicates_removed
)

print(
    "Reconciled total:",
    reconciled_count
)

In [0]:
assert (
    reconciled_count
    == bronze_count
), (
    f"Final reconciliation failed: "
    f"{silver_count} Silver + "
    f"{quarantine_count} Quarantine + "
    f"{duplicates_removed} duplicates "
    f"!= {bronze_count} Bronze"
)

print(
    "✅ FINAL SENSOR RECONCILIATION PASSED"
)

In [0]:
hot_records = (
    quarantine_sensor_table
    .filter(
        F.col("temperature_c") == "hot"
    )
)

display(hot_records)

In [0]:
hot_count = hot_records.count()

print(
    "Nonnumeric 'hot' records quarantined:",
    hot_count
)

assert (
    hot_count == 1
), (
    f"Expected 1 'hot' record in quarantine, "
    f"found {hot_count}"
)

print(
    "✅ Nonnumeric temperature test passed"
)

In [0]:
unknown_shipment_records = (
    quarantine_sensor_table
    .filter(
        F.col("shipment_id") == "SH999"
    )
)

display(
    unknown_shipment_records
)

In [0]:
unknown_count = (
    unknown_shipment_records.count()
)

print(
    "Unknown shipment records quarantined:",
    unknown_count
)

assert (
    unknown_count == 1
), (
    f"Expected 1 SH999 record, "
    f"found {unknown_count}"
)

print(
    "✅ Unknown shipment test passed"
)

In [0]:
display(
    unknown_shipment_records.select(
        "event_id",
        "shipment_id",
        "rejection_reason"
    )
)

In [0]:
missing_event_id_records = (
    quarantine_sensor_table
    .filter(
        F.col("event_id").isNull()
    )
)

display(
    missing_event_id_records
)

In [0]:
missing_event_id_count = (
    missing_event_id_records.count()
)

print(
    "Missing event ID records quarantined:",
    missing_event_id_count
)

assert (
    missing_event_id_count == 1
), (
    f"Expected 1 missing event_id record, "
    f"found {missing_event_id_count}"
)

print(
    "✅ Missing event ID test passed"
)

In [0]:
bronze_e015 = (
    bronze_sensor
    .filter(
        F.col("event_id") == "E015"
    )
)

display(bronze_e015)

In [0]:
bronze_e015_count = (
    bronze_e015.count()
)

print(
    "E015 occurrences in Bronze:",
    bronze_e015_count
)

In [0]:
silver_e015 = (
    silver_sensor_table
    .filter(
        F.col("event_id") == "E015"
    )
)

display(silver_e015)

In [0]:
silver_e015_count = (
    silver_e015.count()
)

print(
    "E015 occurrences in Silver:",
    silver_e015_count
)

assert (
    silver_e015_count == 1
), (
    f"Expected exactly 1 E015 in Silver, "
    f"found {silver_e015_count}"
)

print(
    "✅ Duplicate E015 test passed"
)

In [0]:
e004_silver = (
    silver_sensor_table
    .filter(
        F.col("event_id") == "E004"
    )
)

display(e004_silver)

In [0]:
e004_silver_count = (
    e004_silver.count()
)

print(
    "E004 records in Silver:",
    e004_silver_count
)

assert (
    e004_silver_count == 1
), (
    f"E004 should exist exactly once in Silver, "
    f"found {e004_silver_count}"
)

print(
    "✅ E004 survived technical validation"
)

In [0]:
e004_quarantine_count = (
    quarantine_sensor_table
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)

print(
    "E004 records in quarantine:",
    e004_quarantine_count
)

assert (
    e004_quarantine_count == 0
), (
    "E004 was incorrectly quarantined"
)

print(
    "✅ E004 was NOT incorrectly quarantined"
)

In [0]:
boundary_records = (
    silver_sensor_table
    .filter(
        F.col("event_id").isin(
            "E003",
            "E007"
        )
    )
    .select(
        "event_id",
        "shipment_id",
        "temperature_c",
        "event_time"
    )
)

display(boundary_records)

In [0]:
boundary_count = (
    boundary_records.count()
)

print(
    "Boundary records in Silver:",
    boundary_count
)

assert (
    boundary_count == 2
), (
    f"Expected both E003 and E007 in Silver, "
    f"found {boundary_count}"
)

print(
    "✅ Boundary records survived Silver validation"
)

In [0]:
silver_sensor_table.printSchema()

In [0]:
from pyspark.sql import functions as F

NS = "training_lab_classic.cold_chain"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

print("===== SILVER SENSOR SCHEMA =====")

silver_sensor_table.printSchema()

schema_dict = {
    field.name: field.dataType.simpleString()
    for field in silver_sensor_table.schema.fields
}

print("")
print("Detected column types:")

for column_name, data_type in schema_dict.items():
    print(f"{column_name}: {data_type}")

assert "temperature_c" in schema_dict, (
    "FAILED: temperature_c column does not exist in Silver"
)

assert "event_time" in schema_dict, (
    "FAILED: event_time column does not exist in Silver"
)

assert schema_dict["temperature_c"] == "double", (
    f"FAILED: Expected temperature_c to be double, "
    f"but found {schema_dict['temperature_c']}"
)

assert schema_dict["event_time"] == "timestamp", (
    f"FAILED: Expected event_time to be timestamp, "
    f"but found {schema_dict['event_time']}"
)

print("")
print("✅ Silver schema validation passed")

In [0]:
from pyspark.sql import functions as F

NS = "training_lab_classic.cold_chain"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

silver_null_event_ids = (
    silver_sensor_table
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

print("===== NULL EVENT ID CHECK =====")
print("Null event IDs in Silver:", silver_null_event_ids)

assert silver_null_event_ids == 0, (
    f"FAILED: Silver contains "
    f"{silver_null_event_ids} records "
    f"with NULL event_id"
)

print("✅ Silver contains no NULL event IDs")

In [0]:
from pyspark.sql import functions as F

NS = "training_lab_classic.cold_chain"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

sh999_records = (
    silver_sensor_table
    .filter(
        F.col("shipment_id") == "SH999"
    )
)

display(sh999_records)

sh999_count = sh999_records.count()

print("===== UNKNOWN SHIPMENT CHECK =====")
print("SH999 records in Silver:", sh999_count)

assert sh999_count == 0, (
    f"FAILED: Found "
    f"{sh999_count} SH999 records in Silver"
)

print("✅ Unknown shipment SH999 does not exist in Silver")

In [0]:
from pyspark.sql import functions as F

NS = "training_lab_classic.cold_chain"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

duplicate_ids_in_silver = (
    silver_sensor_table
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

print("===== DUPLICATE EVENT ID CHECK =====")

display(duplicate_ids_in_silver)

duplicate_id_count = (
    duplicate_ids_in_silver.count()
)

print(
    "Number of duplicate event IDs remaining in Silver:",
    duplicate_id_count
)

assert duplicate_id_count == 0, (
    f"FAILED: Silver contains "
    f"{duplicate_id_count} duplicated event IDs"
)

print("✅ Silver contains no duplicate event IDs")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

NS = "training_lab_classic.cold_chain"

# =========================================================
# LOAD TABLES
# =========================================================

bronze = spark.table(
    f"{NS}.bronze_sensor_events"
)

silver = spark.table(
    f"{NS}.silver_sensor_events"
)

quarantine = spark.table(
    f"{NS}.quarantine_sensor_events"
)

# =========================================================
# BASIC TABLE COUNTS
# =========================================================

bronze_count = bronze.count()

silver_count = silver.count()

quarantine_count = quarantine.count()

# =========================================================
# QUARANTINE TESTS
# =========================================================

hot_count = (
    quarantine
    .filter(
        F.col("temperature_c") == "hot"
    )
    .count()
)

unknown_shipment_count = (
    quarantine
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)

missing_event_id_count = (
    quarantine
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

# =========================================================
# E015 DUPLICATE TEST
# =========================================================

e015_bronze_count = (
    bronze
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)

e015_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)

# =========================================================
# E004 VALID OUT-OF-RANGE READING TEST
# =========================================================

e004_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)

e004_quarantine_count = (
    quarantine
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)

# =========================================================
# SILVER NULL-ID TEST
# =========================================================

silver_null_ids = (
    silver
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

# =========================================================
# SILVER UNKNOWN-SHIPMENT TEST
# =========================================================

silver_unknown_shipments = (
    silver
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)

# =========================================================
# SILVER DUPLICATE TEST
# =========================================================

silver_duplicate_ids = (
    silver
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

# =========================================================
# BOUNDARY READING TESTS
# =========================================================

e003_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E003"
    )
    .count()
)

e007_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E007"
    )
    .count()
)

# =========================================================
# SCHEMA VALIDATION
# =========================================================

schema_dict = {
    field.name: field.dataType.simpleString()
    for field in silver.schema.fields
}

temperature_type = schema_dict.get(
    "temperature_c"
)

event_time_type = schema_dict.get(
    "event_time"
)

# =========================================================
# PRINT REPORT
# =========================================================

print("")
print("==============================================")
print("       SENSOR SILVER VALIDATION REPORT")
print("==============================================")

print("")
print("TABLE COUNTS")
print("-------------------------------")

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# ---------------------------------------------------------
# LOAD QUARANTINE TABLE
# ---------------------------------------------------------

quarantine_sensor_table = spark.table(
    f"{NS}.quarantine_sensor_events"
)

# ---------------------------------------------------------
# DISPLAY QUARANTINED RECORDS
# ---------------------------------------------------------

print("==============================================")
print("       QUARANTINED SENSOR RECORDS")
print("==============================================")

display(
    quarantine_sensor_table
    .select(
        "event_id",
        "shipment_id",
        "sensor_id",
        "event_time",
        "temperature_c",
        "rejection_reason",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
    .orderBy(
        "rejection_reason",
        "event_id"
    )
)

# ---------------------------------------------------------
# COUNT QUARANTINED RECORDS
# ---------------------------------------------------------

quarantine_count = quarantine_sensor_table.count()

print("")
print("Total quarantined records:", quarantine_count)

# ---------------------------------------------------------
# COUNT BY REJECTION REASON
# ---------------------------------------------------------

print("")
print("==============================================")
print("       QUARANTINE REASONS")
print("==============================================")

quarantine_reason_summary = (
    quarantine_sensor_table
    .groupBy("rejection_reason")
    .count()
    .orderBy("rejection_reason")
)

display(quarantine_reason_summary)

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# ---------------------------------------------------------
# LOAD SILVER TABLE
# ---------------------------------------------------------

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

# ---------------------------------------------------------
# DISPLAY CLEAN SILVER RECORDS
# ---------------------------------------------------------

print("==============================================")
print("       CLEAN SILVER SENSOR RECORDS")
print("==============================================")

display(
    silver_sensor_table
    .select(
        "event_id",
        "shipment_id",
        "sensor_id",
        "event_time",
        "temperature_c",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
    .orderBy(
        "event_time",
        "event_id"
    )
)

# ---------------------------------------------------------
# COUNT SILVER RECORDS
# ---------------------------------------------------------

silver_count = silver_sensor_table.count()

print("")
print("Total Silver sensor records:", silver_count)

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# ---------------------------------------------------------
# LOAD TABLES
# ---------------------------------------------------------

bronze_sensor_table = spark.table(
    f"{NS}.bronze_sensor_events"
)

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

quarantine_sensor_table = spark.table(
    f"{NS}.quarantine_sensor_events"
)

# ---------------------------------------------------------
# COUNTS
# ---------------------------------------------------------

bronze_count = bronze_sensor_table.count()

silver_count = silver_sensor_table.count()

quarantine_count = quarantine_sensor_table.count()

# ---------------------------------------------------------
# PRINT SUMMARY
# ---------------------------------------------------------

print("==============================================")
print("       MEDALLION SENSOR SUMMARY")
print("==============================================")

print("")
print("Bronze records     :", bronze_count)
print("Silver records     :", silver_count)
print("Quarantine records :", quarantine_count)

print("")
print("==============================================")

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

# ---------------------------------------------------------
# NULL EVENT ID
# ---------------------------------------------------------

null_event_id_count = (
    silver_sensor_table
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

# ---------------------------------------------------------
# UNKNOWN SHIPMENT
# ---------------------------------------------------------

unknown_shipment_count = (
    silver_sensor_table
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)

# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------

print("==============================================")
print("        SILVER INVALID-RECORD CHECK")
print("==============================================")

print("")
print(
    "NULL event_id records in Silver:",
    null_event_id_count
)

print(
    "SH999 records in Silver:",
    unknown_shipment_count
)

# ---------------------------------------------------------
# ASSERT
# ---------------------------------------------------------

assert null_event_id_count == 0, (
    f"FAILED: Found {null_event_id_count} "
    f"NULL event IDs in Silver"
)

assert unknown_shipment_count == 0, (
    f"FAILED: Found {unknown_shipment_count} "
    f"SH999 records in Silver"
)

print("")
print("✅ Invalid records are correctly excluded from Silver")

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

quarantine_sensor_table = spark.table(
    f"{NS}.quarantine_sensor_events"
)

hot_records = (
    quarantine_sensor_table
    .filter(
        F.col("temperature_c") == "hot"
    )
)

print("==============================================")
print("       NONNUMERIC TEMPERATURE CHECK")
print("==============================================")

display(
    hot_records.select(
        "event_id",
        "shipment_id",
        "sensor_id",
        "event_time",
        "temperature_c",
        "rejection_reason"
    )
)

hot_count = hot_records.count()

print("")
print("'hot' records in Quarantine:", hot_count)

assert hot_count == 1, (
    f"FAILED: Expected exactly one 'hot' record "
    f"in Quarantine, found {hot_count}"
)

print("")
print("✅ Nonnumeric temperature correctly quarantined")

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

quarantine_sensor_table = spark.table(
    f"{NS}.quarantine_sensor_events"
)

# ---------------------------------------------------------
# CHECK SILVER
# ---------------------------------------------------------

e004_silver = (
    silver_sensor_table
    .filter(
        F.col("event_id") == "E004"
    )
)

print("E004 record in Silver:")

display(
    e004_silver.select(
        "event_id",
        "shipment_id",
        "sensor_id",
        "event_time",
        "temperature_c"
    )
)

e004_silver_count = (
    e004_silver.count()
)

# ---------------------------------------------------------
# CHECK

In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD BRONZE AND SILVER TABLES
# =========================================================

bronze_sensor_table = spark.table(
    f"{NS}.bronze_sensor_events"
)

silver_sensor_table = spark.table(
    f"{NS}.silver_sensor_events"
)

# =========================================================
# FIND E015 IN BRONZE
# =========================================================

bronze_e015 = (
    bronze_sensor_table
    .filter(
        F.col("event_id") == "E015"
    )
)

print("==============================================")
print("           E015 RECORDS IN BRONZE")
print("==============================================")

display(
    bronze_e015
)

# Count E015 records in Bronze
bronze_e015_count = bronze_e015.count()

print("")
print(
    "E015 count in Bronze:",
    bronze_e015_count
)

# =========================================================
# FIND E015 IN SILVER
# =========================================================

silver_e015 = (
    silver_sensor_table
    .filter(
        F.col("event_id") == "E015"
    )
)

print("")
print("==============================================")
print("           E015 RECORDS IN SILVER")
print("==============================================")

display(
    silver_e015
)

# Count E015 records in Silver
silver_e015_count = silver_e015.count()

print("")
print(
    "E015 count in Silver:",
    silver_e015_count
)

# =========================================================
# VALIDATE THE RESULT
# =========================================================

print("")
print("==============================================")
print("           DEDUPLICATION RESULT")
print("==============================================")

print(
    "Bronze E015 count:",
    bronze_e015_count
)

print(
    "Silver E015 count:",
    silver_e015_count
)

# The generated fixture contains E015 twice in Bronze.
assert bronze_e015_count == 2, (
    f"FAILED: Expected E015 to appear 2 times "
    f"in Bronze, but found {bronze_e015_count}"
)

# Silver should contain only one surviving E015.
assert silver_e015_count == 1, (
    f"FAILED: Expected E015 to appear exactly "
    f"1 time in Silver, but found {silver_e015_count}"
)

print("")
print("✅ E015 deduplication successful")

print("")
print("Bronze : E015 appears twice")
print("Silver : E015 appears once")

print("")
print("==============================================")
print("             STEP 45 PASSED")
print("==============================================")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# STEP 46: FINAL SENSOR SILVER CERTIFICATE
# =========================================================

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

print("Namespace:", NS)


# ---------------------------------------------------------
# 2. LOAD BRONZE, SILVER AND QUARANTINE TABLES
# ---------------------------------------------------------

bronze = spark.table(
    f"{NS}.bronze_sensor_events"
)

silver = spark.table(
    f"{NS}.silver_sensor_events"
)

quarantine = spark.table(
    f"{NS}.quarantine_sensor_events"
)


# ---------------------------------------------------------
# 3. BASIC TABLE COUNTS
# ---------------------------------------------------------

bronze_count = bronze.count()

silver_count = silver.count()

quarantine_count = quarantine.count()


# ---------------------------------------------------------
# 4. VERIFY "hot" WAS QUARANTINED
# ---------------------------------------------------------

hot_quarantine_count = (
    quarantine
    .filter(
        F.col("temperature_c") == "hot"
    )
    .count()
)


# ---------------------------------------------------------
# 5. VERIFY SH999 WAS QUARANTINED
# ---------------------------------------------------------

unknown_shipment_quarantine_count = (
    quarantine
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)


# ---------------------------------------------------------
# 6. VERIFY NULL EVENT ID WAS QUARANTINED
# ---------------------------------------------------------

missing_event_id_quarantine_count = (
    quarantine
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)


# ---------------------------------------------------------
# 7. VERIFY SILVER HAS NO NULL EVENT IDS
# ---------------------------------------------------------

null_event_ids_in_silver = (
    silver
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)


# ---------------------------------------------------------
# 8. VERIFY SILVER HAS NO UNKNOWN SH999
# ---------------------------------------------------------

unknown_shipments_in_silver = (
    silver
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)


# ---------------------------------------------------------
# 9. VERIFY SILVER HAS NO DUPLICATE EVENT IDS
# ---------------------------------------------------------

duplicate_ids_in_silver = (
    silver
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


# ---------------------------------------------------------
# 10. VERIFY E015 DUPLICATION / DEDUPLICATION
# ---------------------------------------------------------

e015_bronze_count = (
    bronze
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)

e015_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)


# ---------------------------------------------------------
# 11. VERIFY E004 SURVIVED INTO SILVER
# ---------------------------------------------------------

e004_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)

e004_quarantine_count = (
    quarantine
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)


# ---------------------------------------------------------
# 12. VERIFY BOUNDARY RECORD E003
# ---------------------------------------------------------

e003_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E003"
    )
    .count()
)


# ---------------------------------------------------------
# 13. VERIFY BOUNDARY RECORD E007
# ---------------------------------------------------------

e007_silver_count = (
    silver
    .filter(
        F.col("event_id") == "E007"
    )
    .count()
)


# ---------------------------------------------------------
# 14. VALIDATE SILVER DATA TYPES
# ---------------------------------------------------------

schema_dict = {
    field.name: field.dataType.simpleString()
    for field in silver.schema.fields
}

temperature_type = schema_dict.get(
    "temperature_c"
)

event_time_type = schema_dict.get(
    "event_time"
)


# ---------------------------------------------------------
# 15. DISPLAY VALIDATION REPORT
# ---------------------------------------------------------

print("")
print("================================================")
print("       FINAL SENSOR SILVER CERTIFICATE")
print("================================================")

print("")
print("TABLE COUNTS")
print("------------------------------------------------")
print("Bronze rows                     :", bronze_count)
print("Silver rows                     :", silver_count)
print("Quarantine rows                 :", quarantine_count)

print("")
print("QUARANTINE CHECKS")
print("------------------------------------------------")
print("'hot' quarantined               :", hot_quarantine_count)
print("SH999 quarantined               :", unknown_shipment_quarantine_count)
print("NULL event_id quarantined       :", missing_event_id_quarantine_count)

print("")
print("SILVER QUALITY CHECKS")
print("------------------------------------------------")
print("NULL event IDs in Silver        :", null_event_ids_in_silver)
print("SH999 records in Silver         :", unknown_shipments_in_silver)
print("Duplicate event IDs in Silver   :", duplicate_ids_in_silver)

print("")
print("DEDUPLICATION CHECK")
print("------------------------------------------------")
print("E015 occurrences in Bronze      :", e015_bronze_count)
print("E015 occurrences in Silver      :", e015_silver_count)

print("")
print("VALID OUT-OF-RANGE READING CHECK")
print("------------------------------------------------")
print("E004 occurrences in Silver      :", e004_silver_count)
print("E004 occurrences in Quarantine  :", e004_quarantine_count)

print("")
print("BOUNDARY VALUE CHECKS")
print("------------------------------------------------")
print("E003 occurrences in Silver      :", e003_silver_count)
print("E007 occurrences in Silver      :", e007_silver_count)

print("")
print("SILVER DATA TYPES")
print("------------------------------------------------")
print("temperature_c type              :", temperature_type)
print("event_time type                 :", event_time_type)

print("")
print("================================================")


# ---------------------------------------------------------
# 16. ASSERT EXPECTED RESULTS
# ---------------------------------------------------------

assert bronze_count == 20, (
    f"FAILED: Expected 20 Bronze records, "
    f"but found {bronze_count}"
)

assert hot_quarantine_count == 1, (
    f"FAILED: Expected exactly 1 'hot' record "
    f"in Quarantine, but found {hot_quarantine_count}"
)

assert unknown_shipment_quarantine_count == 1, (
    f"FAILED: Expected exactly 1 SH999 record "
    f"in Quarantine, but found "
    f"{unknown_shipment_quarantine_count}"
)

assert missing_event_id_quarantine_count == 1, (
    f"FAILED: Expected exactly 1 missing event_id "
    f"record in Quarantine, but found "
    f"{missing_event_id_quarantine_count}"
)

assert null_event_ids_in_silver == 0, (
    f"FAILED: Silver contains "
    f"{null_event_ids_in_silver} NULL event IDs"
)

assert unknown_shipments_in_silver == 0, (
    f"FAILED: Silver contains "
    f"{unknown_shipments_in_silver} "
    f"unknown shipment records"
)

assert duplicate_ids_in_silver == 0, (
    f"FAILED: Silver contains "
    f"{duplicate_ids_in_silver} duplicate event IDs")